## Import

In [103]:
import pandas as pd
pd.set_option('display.max_columns', None)

df = pd.read_csv('../../data/arm-ready.csv')
df.head()

,ISRC,Track,Album Name,Artist,Release Date,All Time Rank,Track Score,Spotify Streams,Spotify Popularity,YouTube Views,YouTube Likes,TikTok Posts,TikTok Likes,TikTok Views,AirPlay Spins,Pandora Streams,Pandora Track Stations,Shazam Counts,Explicit Track,Length,Releases,Genres,Registration Country,Playlist Probability,High Playlist Probability,Genre,Followers
0,QM24S2402528,MILLION DOLLAR BABY,Million Dollar Baby - Single,Tommy Richman,4/26/2024,1,725.4,390470936,92,84274754,1713126,5767700,651565900,5332281936,40975,18004655,22931,2669262,0,155.1510,1,[],United States,0.734589,1,NaN,847504.0
1,USUG12400910,Not Like Us,Not Like Us,Kendrick Lamar,5/4/2024,2,545.9,323703884,92,116347040,3486739,674700,35223547,208339025,40778,7780028,28444,1118279,1,274.1920,5,"['hip hop', 'producer tag - dj mustard']",United States,0.721077,1,"hip hop, west coast hip hop",38589324.0
2,QZJ842400387,i like the way you kiss me,I like the way you kiss me,Artemas,3/19/2024,3,538.4,601309283,92,122599116,2228730,3025400,275154237,3369120610,74333,5022621,5639,5285340,0,143.1865,8,['synth-pop'],United States,0.771418,1,NaN,1133424.0
3,USSM12209777,Flowers,Flowers - Single,Miley Cyrus,1/12/2023,4,444.9,2031280633,85,1096100899,10629796,7189811,1078757968,14603725994,1474799,190260277,203384,11822942,0,200.4530,30,"['yacht rock', 'alternative pop', 'pop rock', ...",United States,0.799485,1,NaN,25620499.0
4,USUG12403398,Houdini,Houdini,Eminem,5/31/2024,5,423.3,107034922,88,77373957,3670188,16400,26617282,266222834,12185,4493884,7006,457017,1,227.0320,20,[],United States,0.721544,1,"rap, hip hop",97970063.0


## Augment

In [104]:
# TikTok
df['TikTok Views per Post'] = df['TikTok Views'] / df['TikTok Posts'] #
df['TikTok Impact'] = df['TikTok Views per Post'] / (df['YouTube Views'] + 1) #
df['TikTok View-to-Like Ratio'] = df['TikTok Views'] / (df['TikTok Likes'] + 1) #
df['TikTok Likes per Post'] = df['TikTok Likes'] / (df['TikTok Posts'] + 1) #

# YouTube
df['YouTube View-to-Like Ratio'] = df['YouTube Views'] / (df['YouTube Likes'] + 1) #

# Shazam
df['Shazam Conversion Rate'] = df['Shazam Counts'] / (df['YouTube Views'] + df['TikTok Views'] + 1) #

## Reduce Genres

In [105]:
genre_map = {
    'Pop': ['pop', 'dance pop', 'pop punk', 'pop rock', 'electropop', 'latin pop', 'k-pop', 'j-pop', 'bedroom pop', 'acoustic pop', 'indie pop', 'pop urbano', 'colombian pop', 'soft pop'],
    'Rock': ['rock', 'alternative rock', 'classic rock', 'garage rock', 'hard rock', 'metal', 'punk', 'emo', 'grunge', 'post-grunge', 'psychedelic rock', 'ska', 'ska punk', 'indie rock'],
    'Rap': ['rap', 'hip hop', 'trap', 'drill', 'grime', 'uk drill', 'west coast hip hop', 'east coast hip hop', 'underground hip hop', 'emo rap', 'rage rap', 'gangster rap', 'latin hip hop', 'experimental hip hop'],
    'R&B': ['r&b', 'neo soul', 'alternative r&b', 'motown', 'soul', 'latin r&b'],
    'Electronic': ['edm', 'electronic', 'house', 'techno', 'electro', 'dubstep', 'progressive house', 'melodic house', 'trance', 'synthwave', 'future house', 'eurodance'],
    'Country': ['country', 'country rock', 'country pop', 'acoustic country', 'bluegrass', 'red dirt'],
    'Reggaeton': ['reggaeton', 'reggaeton chileno', 'urbano latino', 'latin urban', 'neoperreo'],
    'Latin': ['latin', 'bachata', 'salsa', 'cumbia', 'ranchera', 'vallenato', 'flamenco', 'latin folk', 'tropical house']
}
def map_genre(genre_item):
    """Maps a single subgenre string to a major genre."""
    if not genre_item or pd.isna(genre_item): # Handle empty strings or NaN
        return 'Other'

    genre_item = str(genre_item).lower() # Ensure it's a lowercase string
    for broad_genre, subgenres in genre_map.items():
        if genre_item in subgenres:
            return broad_genre
    return 'Other'

## Process Dataset

In [106]:
# -- Genre
# extract sub genres and explode them
df = df.rename(columns={'Genre': 'Sub Genre'})
df['Sub Genre'] = df['Sub Genre'].fillna('')
df['Sub Genre'] = df['Sub Genre'].str.split(', ')
df = df.explode('Sub Genre')

df['Genre'] = df['Sub Genre'].apply(map_genre)
df = df.drop(columns=['Sub Genre', 'Genres'])

# groupby isrc on mode(genre) 
aggregation_functions = {
    'Genre': lambda x: x.mode()[0] if not x.mode().empty else None
}

other_cols = [col for col in df.columns if col not in ['ISRC', 'Genre']]
for col in other_cols:
    aggregation_functions[col] = 'first'
    
df_agg = df.groupby('ISRC').agg(aggregation_functions)
df = df_agg.reset_index()


# -- Release Date
current_date = pd.Timestamp.now().normalize() 

df['Release Datetime'] = pd.to_datetime(
    df['Release Date'],
    errors='coerce',
    dayfirst=False 
)

valid_dates_mask = df['Release Datetime'].notna()
df['Days Since Release'] = np.nan

release_times = df.loc[valid_dates_mask, 'Release Datetime']
days_since_release = (current_date - release_times).dt.days

col_index = df.columns.get_loc('Days Since Release')
row_indices = np.where(valid_dates_mask)[0]
df.iloc[row_indices, col_index] = days_since_release.to_numpy()

df.drop(columns=['Release Datetime', 'Release Date'], inplace=True)


# -- Label
df['Label Prob'] = df[['Playlist Probability']]
df['Label'] = df['High Playlist Probability']
df = df.drop(columns=['High Playlist Probability', 'Playlist Probability'])


# -- Output
df = df.sort_values(by='All Time Rank').reset_index().drop(columns=['index'])

/tmp/ipykernel_658/1390763458.py:41: FutureWarning: In a future version, `df.iloc[:, i] = newvals` will attempt to set the values inplace instead of always setting a new array. To retain the old behavior, use either `df[df.columns[i]] = newvals` or, if columns are non-unique, `df.isetitem(i, newvals)`
  df.iloc[row_indices, col_index] = days_since_release.to_numpy()


## Result

In [107]:
df.head()

,ISRC,Genre,Track,Album Name,Artist,All Time Rank,Track Score,Spotify Streams,Spotify Popularity,YouTube Views,YouTube Likes,TikTok Posts,TikTok Likes,TikTok Views,AirPlay Spins,Pandora Streams,Pandora Track Stations,Shazam Counts,Explicit Track,Length,Releases,Registration Country,Followers,TikTok Views per Post,TikTok Impact,TikTok View-to-Like Ratio,TikTok Likes per Post,YouTube View-to-Like Ratio,Shazam Conversion Rate,Days Since Release,Label Prob,Label
0,QM24S2402528,Other,MILLION DOLLAR BABY,Million Dollar Baby - Single,Tommy Richman,1,725.4,390470936,92,84274754,1713126,5767700,651565900,5332281936,40975,18004655,22931,2669262,0,155.1510,1,United States,847504.0,924.507505,0.000011,8.183795,112.968044,49.193524,0.000493,344,0.734589,1
1,USUG12400910,Rap,Not Like Us,Not Like Us,Kendrick Lamar,2,545.9,323703884,92,116347040,3486739,674700,35223547,208339025,40778,7780028,28444,1118279,1,274.1920,5,United States,38589324.0,308.787646,0.000003,5.914765,52.206158,33.368430,0.003444,336,0.721077,1
2,QZJ842400387,Other,i like the way you kiss me,I like the way you kiss me,Artemas,3,538.4,601309283,92,122599116,2228730,3025400,275154237,3369120610,74333,5022621,5639,5285340,0,143.1865,8,United States,1133424.0,1113.611625,0.000009,12.244480,90.948022,55.008485,0.001514,382,0.771418,1
3,USSM12209777,Other,Flowers,Flowers - Single,Miley Cyrus,4,444.9,2031280633,85,1096100899,10629796,7189811,1078757968,14603725994,1474799,190260277,203384,11822942,0,200.4530,30,United States,25620499.0,2031.169664,0.000002,13.537537,150.039802,103.115883,0.000753,814,0.799485,1
4,USUG12403398,Rap,Houdini,Houdini,Eminem,5,423.3,107034922,88,77373957,3670188,16400,26617282,266222834,12185,4493884,7006,457017,1,227.0320,20,United States,97970063.0,16233.099634,0.000210,10.001879,1622.906042,21.081736,0.001330,309,0.721544,1


In [108]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4572 entries, 0 to 4571
Data columns (total 32 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   ISRC                        4572 non-null   object 
 1   Genre                       4572 non-null   object 
 2   Track                       4572 non-null   object 
 3   Album Name                  4572 non-null   object 
 4   Artist                      4572 non-null   object 
 5   All Time Rank               4572 non-null   int64  
 6   Track Score                 4572 non-null   float64
 7   Spotify Streams             4572 non-null   int64  
 8   Spotify Popularity          4572 non-null   int64  
 9   YouTube Views               4572 non-null   int64  
 10  YouTube Likes               4572 non-null   int64  
 11  TikTok Posts                4572 non-null   int64  
 12  TikTok Likes                4572 non-null   int64  
 13  TikTok Views                4572 

In [109]:
df['Followers'] = df['Followers'].fillna(0)
df['Followers'].isnull().sum()

0

In [110]:
df.to_csv('../../data/model-ready.csv', encoding='utf-8', index=False)